<a href="https://colab.research.google.com/github/NiclasFenton-Wiegleb/rubber-duck-games/blob/main/RubberDuckGames.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating RAG Knowledge Graph

In order for the app to have accurate knowledge of the Godot documentation and how to address user queries in this context, relevent information needs to be injected into the model prompt. One of the best ways to do this is Retrieval Augemented Generation (RAG). Knowledge Graphs provide additional benefit over vanilla RAG, as they preserve relationships between information and capture a more accurate picture of the knowledge. The final workflow of the app should look as follows:

```
User query
    │
    ▼
Embed query (bge-small)
    │
    ▼
FAISS top-k chunk retrieval  ──→  Entry point nodes in KG
    │
    ▼
Graph traversal (1-2 hops)
  - Pull connected entity nodes
  - Pull sibling/parent section chunks
  - Pull cross-referenced chunks
    │
    ▼
Re-rank + deduplicate retrieved context
    │
    ▼
Build prompt with context + user query
    │
    ▼
Fine-tuned SLM generates answer

```



## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install beautifulsoup4 lxml spacy sentence-transformers networkx huggingface_hub tqdm
!python -m spacy download en_core_web_sm
!pip install faiss-gpu-cu12

# faiss-gpu for Colab GPU runtime; swap to faiss-cpu if on CPU
print('Dependencies installed.')

In [ ]:
%%capture
!git clone --depth 1 https://github.com/cromerc/godot-offline-docs /content/godot-docs

In [ ]:
import os, subprocess, glob
from tqdm.auto import tqdm
import sys
from google.colab import userdata

from bs4 import BeautifulSoup
import re, json
from pathlib import Path
from tqdm.auto import tqdm

import spacy, networkx as nx, pickle
from collections import defaultdict

import torch
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

from huggingface_hub import HfApi

In [ ]:
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## Step 2 — Configuration

In [ ]:
HF_REPO_ID      = "niclasfw/rubber-duck-games"
HF_REPO_TYPE    = "space"                        # "space" | "dataset" | "model"

# Which Godot doc versions to process (subdirs under html/)
# Set to None to process ALL versions found
VERSIONS_TO_PROCESS = ["3.5", "3.4"]            # or None for all

# Chunking
CHUNK_SIZE      = 400    # target tokens per chunk
CHUNK_OVERLAP   = 60     # overlap tokens between chunks

# Embedding model (small, fast, high quality)
EMBED_MODEL     = "BAAI/bge-small-en-v1.5"
EMBED_BATCH     = 64     # reduce if OOM

# Output paths (local in Colab, then uploaded)
OUT_DIR         = "/content/rag_artifacts"

# Remote paths inside HF storage
HF_DATA_DIR     = "data"                        # artifacts will be under data/

# ───────────────────────────────────────────────────────────────────────────────

os.makedirs(f"{OUT_DIR}/chunks",  exist_ok=True)
os.makedirs(f"{OUT_DIR}/faiss",   exist_ok=True)
os.makedirs(f"{OUT_DIR}/kg",      exist_ok=True)
print('Config OK. Output dir:', OUT_DIR)

Config OK. Output dir: /content/rag_artifacts


## Step 3 — Clone the Docs Repo

In [ ]:
DOCS_HTML_ROOT  = "/content/godot-docs-html"
DOCS_SRC_ROOT   = "/content/godot-docs-src"
os.makedirs(DOCS_HTML_ROOT, exist_ok=True)
os.makedirs(DOCS_SRC_ROOT,  exist_ok=True)

VERSION_BRANCHES = {
    "3.5": "3.5",
    "3.4": "3.4",
    "3.3": "3.3",
    "3.2": "3.2",
    "3.1": "3.1",
    "3.0": "3.0",
}

# Sphinx version known to build all godot 3.x branches cleanly.
# The 3.x requirements.txt doesn't pin Sphinx, so we pin it here
# before installing repo deps to prevent the applehelp version conflict.
SPHINX_VERSION = "sphinx==5.3.0"

if VERSIONS_TO_PROCESS is None:
    versions = list(VERSION_BRANCHES.keys())
else:
    versions = [v for v in VERSIONS_TO_PROCESS if v in VERSION_BRANCHES]
    missing  = [v for v in VERSIONS_TO_PROCESS if v not in VERSION_BRANCHES]
    if missing:
        print(f"WARNING: no branch known for versions: {missing}")

print(f"Will build: {versions}\n")

# ── Install Sphinx (pinned) once up front ──────────────────────────────────────
print(f"Installing {SPHINX_VERSION} ...")
subprocess.run(
    ["pip", "install", "-q", SPHINX_VERSION],
    check=True
)
print("Sphinx installed.\n")

versions_built = []

for version in versions:
    branch   = VERSION_BRANCHES[version]
    src_dir  = os.path.join(DOCS_SRC_ROOT, version)
    html_dir = os.path.join(DOCS_HTML_ROOT, version)

    # ── 1. Clone ────────────────────────────────────────────────────────────
    if os.path.isdir(src_dir):
        print(f"v{version}: source already cloned, skipping.")
    else:
        print(f"Cloning godot-docs @ branch {branch} ...")
        result = subprocess.run([
            "git", "clone", "--depth", "1",
            "--branch", branch, "--single-branch",
            "https://github.com/godotengine/godot-docs.git",
            src_dir,
        ], capture_output=True, text=True)

        if result.returncode != 0:
            print(f"  ERROR cloning v{version}:\n{result.stderr}")
            continue
        print(f"  Cloned OK.")

    # ── 2. Install repo deps WITHOUT re-resolving Sphinx ────────────────────
    req_file = os.path.join(src_dir, "requirements.txt")
    if os.path.isfile(req_file):
        print(f"  Installing deps from requirements.txt ...")
        # --no-deps on sphinx-related packages would be ideal, but the safest
        # approach is to install repo deps then re-pin Sphinx if overwritten.
        result = subprocess.run(
            ["pip", "install", "-q", "-r", req_file],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"  pip error:\n{result.stderr[-500:]}")

        # Re-pin Sphinx in case the requirements pulled a different version
        subprocess.run(
            ["pip", "install", "-q", SPHINX_VERSION],
            capture_output=True
        )
        print(f"  Deps installed, Sphinx re-pinned to {SPHINX_VERSION}.")
    else:
        print(f"  No requirements.txt found, installing known 3.x deps ...")
        subprocess.run([
            "pip", "install", "-q",
            SPHINX_VERSION,
            "sphinx-rtd-theme",
            "sphinx-tabs",
            "sphinx-notfound-page",
            "Pillow",
        ], check=True)

    # ── 3. Build HTML ────────────────────────────────────────────────────────
    if os.path.isdir(html_dir) and any(
        f.endswith(".html")
        for _, _, files in os.walk(html_dir)
        for f in files
    ):
        print(f"v{version}: HTML already built, skipping.")
        versions_built.append(version)
        continue

    os.makedirs(html_dir, exist_ok=True)
    print(f"  Building HTML for v{version} (~3–5 min)...")

    result = subprocess.run(
        [
            "python", "-m", "sphinx",
            "-b", "html",
            "-j", "auto",
            "-q",
            "-D", "language=en",
            src_dir,
            html_dir,
        ],
        capture_output=True, text=True, cwd=src_dir
    )

    if result.returncode != 0:
        print(f"  Sphinx ERROR for v{version}:")
        for line in result.stderr.strip().splitlines()[-40:]:
            print("  ", line)
        continue

    html_count = sum(
        1 for _, _, files in os.walk(html_dir)
        for f in files if f.endswith(".html")
    )
    print(f"  Built OK — {html_count:,} HTML files.")
    versions_built.append(version)

# ── Summary ────────────────────────────────────────────────────────────────────
print(f"\n── Verification ──────────────────────────────────────────────")
versions = versions_built
for version in versions:
    html_count = sum(
        1 for _, _, files in os.walk(os.path.join(DOCS_HTML_ROOT, version))
        for f in files if f.endswith(".html")
    )
    print(f"  v{version}: {html_count:,} HTML files → {DOCS_HTML_ROOT}/{version}/")

print(f"\nProcessing versions: {versions}")

Will build: ['3.5', '3.4']

Installing sphinx==5.3.0 ...
Sphinx installed.

v3.5: source already cloned, skipping.
  Installing deps from requirements.txt ...
  Deps installed, Sphinx re-pinned to sphinx==5.3.0.
  Building HTML for v3.5 (~3–5 min)...
  Built OK — 1,086 HTML files.
v3.4: source already cloned, skipping.
  Installing deps from requirements.txt ...
  Deps installed, Sphinx re-pinned to sphinx==5.3.0.
  Building HTML for v3.4 (~3–5 min)...
  Built OK — 1,057 HTML files.

── Verification ──────────────────────────────────────────────
  v3.5: 1,086 HTML files → /content/godot-docs-html/3.5/
  v3.4: 1,057 HTML files → /content/godot-docs-html/3.4/

Processing versions: ['3.5', '3.4']


## Step 4 — Parse & Chunk HTML

Godot docs are Sphinx-generated HTML. We extract:

 - Clean body text (stripping nav, sidebar, footer)
 - Section headings as metadata
 - Source URL / file path
 - Doc version

In [ ]:
# ── HTML Parser ────────────────────────────────────────────────────────────────

def parse_html_file(filepath: str, version: str) -> list[dict]:
    """
    Parse a single Godot docs HTML file.
    Returns a list of section dicts: {text, heading, doc_title, source_file, version}
    """
    with open(filepath, encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f, "lxml")

    # Remove nav, sidebar, footer, scripts, styles
    for tag in soup.find_all(["nav", "footer", "script", "style",
                               "header", "aside"]):
        tag.decompose()
    for tag in soup.find_all(class_=["headerlink", "toctree-wrapper",
                                      "sphinxsidebar", "related", "footer"]):
        tag.decompose()

    # Page title
    title_tag = soup.find("title")
    doc_title = title_tag.get_text(strip=True) if title_tag else Path(filepath).stem
    doc_title = re.sub(r"\s*[—–-].*$", "", doc_title).strip()  # drop " — Godot Engine docs"

    # Main content area (Sphinx uses <div class="document"> or <div role="main">)
    main = (soup.find("div", role="main")
            or soup.find("div", class_="document")
            or soup.find("article")
            or soup.body)

    if main is None:
        return []

    # Split by sections (h1–h3), carry heading as metadata
    sections = []
    current_heading = doc_title
    current_texts = []

    for el in main.descendants:
        if el.name in ("h1", "h2", "h3"):
            # Save previous section
            text = " ".join(current_texts).strip()
            if text:
                sections.append({
                    "heading": current_heading,
                    "text": text,
                    "doc_title": doc_title,
                    "source_file": str(filepath),
                    "version": version,
                })
            current_heading = el.get_text(strip=True)
            current_texts = []
        elif el.name in ("p", "li", "td", "code", "pre") and el.string:
            t = el.get_text(separator=" ", strip=True)
            if t:
                current_texts.append(t)

    # Last section
    text = " ".join(current_texts).strip()
    if text:
        sections.append({
            "heading": current_heading,
            "text": text,
            "doc_title": doc_title,
            "source_file": str(filepath),
            "version": version,
        })

    return sections


# ── Chunker ────────────────────────────────────────────────────────────────────

def rough_token_count(text: str) -> int:
    """Approximation: 1 token ≈ 4 chars."""
    return len(text) // 4


def chunk_section(section: dict, chunk_size: int, overlap: int) -> list[dict]:
    """
    Split a section's text into overlapping chunks.
    Preserves all metadata fields.
    """
    words = section["text"].split()
    # Convert token targets to word targets (tokens ≈ words * 0.75)
    word_size    = int(chunk_size / 0.75)
    word_overlap = int(overlap / 0.75)

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + word_size, len(words))
        chunk_text = " ".join(words[start:end])
        chunks.append({
            **section,
            "text": chunk_text,
            "chunk_index": len(chunks),
        })
        if end == len(words):
            break
        start += word_size - word_overlap

    return chunks


# ── Process all versions ───────────────────────────────────────────────────────

all_chunks = []
chunk_id   = 0

for version in versions:
    version_dir = os.path.join(DOCS_HTML_ROOT, version)
    html_files  = glob.glob(os.path.join(version_dir, "**/*.html"), recursive=True)
    print(f"\nVersion {version}: {len(html_files)} HTML files")

    for fpath in tqdm(html_files, desc=f"v{version}"):
        sections = parse_html_file(fpath, version)
        for sec in sections:
            for chunk in chunk_section(sec, CHUNK_SIZE, CHUNK_OVERLAP):
                chunk["chunk_id"] = chunk_id
                all_chunks.append(chunk)
                chunk_id += 1

print(f"\nTotal chunks: {len(all_chunks):,}")

# Save chunks to JSONL
chunks_path = f"{OUT_DIR}/chunks/chunks.jsonl"
with open(chunks_path, "w") as f:
    for c in all_chunks:
        f.write(json.dumps(c) + "\n")

print(f"Saved chunks → {chunks_path}")


Version 3.5: 1086 HTML files


v3.5:   0%|          | 0/1086 [00:00<?, ?it/s]


Version 3.4: 1057 HTML files


v3.4:   0%|          | 0/1057 [00:00<?, ?it/s]


Total chunks: 10,713
Saved chunks → /content/rag_artifacts/chunks/chunks.jsonl


## Step 5 — Build the Knowledge Graph

We build a NetworkX graph with four node types:
- doc — per HTML file
- section — per heading within a doc
- chunk — each text chunk
- entity — named entities and Godot-specific terms (classes, nodes, signals)

Edges:
- doc → has_section → section
- section → has_chunk → chunk
- chunk → contains_entity → entity
- entity → co_occurs_with → entity (within the same chunk)
- entity → mentioned_in → chunk (cross-references)

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["parser"])
nlp.max_length = 2_000_000


# ── Godot-specific entity patterns ────────────────────────────────────────────
# These capture PascalCase class names, @keywords, and signal patterns

GODOT_PATTERN = re.compile(
    r"\b([A-Z][a-zA-Z0-9]{2,})\b"        # PascalCase (Node2D, KinematicBody, etc.)
    r"|(@[a-z_]+)"                         # @GDScript, @export, etc.
    r"|(\$[A-Za-z_][A-Za-z0-9_]*)"        # $NodeName shorthand
)

STOP_PASCAL = {"The", "This", "That", "With", "When", "For", "You",
               "See", "Note", "If", "In", "An", "And", "Or"}

def extract_entities(text: str) -> set[str]:
    """Extract both spaCy NER entities and Godot-specific terms."""
    entities = set()

    # spaCy NER (ORG, PRODUCT, PERSON often captures API names)
    doc = nlp(text[:100_000])  # cap for speed
    for ent in doc.ents:
        if ent.label_ in ("ORG", "PRODUCT", "WORK_OF_ART"):
            entities.add(ent.text.strip())

    # Godot-specific regex patterns
    for match in GODOT_PATTERN.finditer(text):
        token = match.group().strip()
        if token and token not in STOP_PASCAL and len(token) > 2:
            entities.add(token)

    return entities


# ── Build graph ───────────────────────────────────────────────────────────────

G = nx.DiGraph()

# Track which chunks mention each entity (for cross-ref edges)
entity_to_chunks: dict[str, list[int]] = defaultdict(list)

print("Building knowledge graph...")

# Group chunks by (version, source_file, heading)
doc_nodes    = {}   # source_file → node id
section_nodes = {}  # (source_file, heading) → node id

for chunk in tqdm(all_chunks, desc="Graph nodes"):
    cid     = chunk["chunk_id"]
    src     = chunk["source_file"]
    heading = chunk["heading"]
    version = chunk["version"]
    title   = chunk["doc_title"]

    # Doc node
    doc_key = (version, src)
    if doc_key not in doc_nodes:
        doc_node_id = f"doc::{version}::{Path(src).stem}"
        G.add_node(doc_node_id, type="doc", title=title,
                   version=version, source_file=src)
        doc_nodes[doc_key] = doc_node_id
    doc_node_id = doc_nodes[doc_key]

    # Section node
    sec_key = (version, src, heading)
    if sec_key not in section_nodes:
        sec_node_id = f"sec::{version}::{Path(src).stem}::{heading[:60]}"
        G.add_node(sec_node_id, type="section", heading=heading,
                   version=version, doc_title=title)
        G.add_edge(doc_node_id, sec_node_id, rel="has_section")
        section_nodes[sec_key] = sec_node_id
    sec_node_id = section_nodes[sec_key]

    # Chunk node
    chunk_node_id = f"chunk::{cid}"
    G.add_node(chunk_node_id, type="chunk", chunk_id=cid,
               text=chunk["text"],
               heading=heading, doc_title=title,
               version=version, source_file=src,
               chunk_index=chunk["chunk_index"])
    G.add_edge(sec_node_id, chunk_node_id, rel="has_chunk")

    # Entity nodes and edges
    entities = extract_entities(chunk["text"])
    entity_list = list(entities)

    for ent in entity_list:
        ent_node_id = f"entity::{ent}"
        if not G.has_node(ent_node_id):
            G.add_node(ent_node_id, type="entity", name=ent)
        G.add_edge(chunk_node_id, ent_node_id, rel="contains_entity")
        G.add_edge(ent_node_id, chunk_node_id, rel="mentioned_in")
        entity_to_chunks[ent].append(cid)

    # Co-occurrence edges between entities in this chunk
    for i in range(len(entity_list)):
        for j in range(i + 1, min(i + 5, len(entity_list))):
            e1 = f"entity::{entity_list[i]}"
            e2 = f"entity::{entity_list[j]}"
            if not G.has_edge(e1, e2):
                G.add_edge(e1, e2, rel="co_occurs_with", weight=1)
            else:
                G[e1][e2]["weight"] = G[e1][e2].get("weight", 1) + 1

print(f"\nGraph stats:")
print(f"  Nodes : {G.number_of_nodes():,}")
print(f"  Edges : {G.number_of_edges():,}")

node_types = {}
for _, data in G.nodes(data=True):
    t = data.get("type", "unknown")
    node_types[t] = node_types.get(t, 0) + 1
for t, count in sorted(node_types.items()):
    print(f"  {t:12s}: {count:,}")

Building knowledge graph...


Graph nodes:   0%|          | 0/10713 [00:00<?, ?it/s]


Graph stats:
  Nodes : 29,165
  Edges : 143,063
  chunk       : 10,713
  doc         : 1,990
  entity      : 5,958
  section     : 10,504


In [ ]:
# Save the graph
kg_path = f"{OUT_DIR}/kg/graph.pkl"
with open(kg_path, "wb") as f:
    pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)

# Also save entity→chunks mapping for fast lookup
e2c_path = f"{OUT_DIR}/kg/entity_to_chunks.json"
with open(e2c_path, "w") as f:
    json.dump(entity_to_chunks, f)

print(f"Graph saved → {kg_path}")
print(f"Entity map  → {e2c_path}")

size_mb = os.path.getsize(kg_path) / 1e6
print(f"Graph file size: {size_mb:.1f} MB")

Graph saved → /content/rag_artifacts/kg/graph.pkl
Entity map  → /content/rag_artifacts/kg/entity_to_chunks.json
Graph file size: 12.7 MB


## Step 6 — Generate Embeddings & Build FAISS Index
We embed every chunk (not entity nodes — they'll be reached via graph traversal). Uses GPU via sentence-transformers if available.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print(f"Loading embedding model: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL, device=device)

# Extract texts and IDs in order (only chunk nodes)
chunk_nodes = [(nid, data) for nid, data in G.nodes(data=True)
               if data.get("type") == "chunk"]
chunk_nodes.sort(key=lambda x: x[1]["chunk_id"])  # ensure deterministic order

node_ids = [nid for nid, _ in chunk_nodes]
texts    = [data["text"] for _, data in chunk_nodes]

print(f"Embedding {len(texts):,} chunks in batches of {EMBED_BATCH}...")

# BGE models work best with a query/passage prefix
texts_with_prefix = [f"passage: {t}" for t in texts]

embeddings = embedder.encode(
    texts_with_prefix,
    batch_size=EMBED_BATCH,
    show_progress_bar=True,
    normalize_embeddings=True,   # cosine similarity via inner product
    convert_to_numpy=True,
)

print(f"Embedding shape: {embeddings.shape}")

Using device: cuda
Loading embedding model: BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding 10,713 chunks in batches of 64...


Batches:   0%|          | 0/168 [00:00<?, ?it/s]

Embedding shape: (10713, 384)


In [ ]:
# Build FAISS index (inner product on normalized vecs = cosine similarity)
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

# Wrap with IDMap so we can use our chunk_ids directly
chunk_ids_array = np.array([data["chunk_id"] for _, data in chunk_nodes], dtype=np.int64)
id_index = faiss.IndexIDMap(index)
id_index.add_with_ids(embeddings, chunk_ids_array)

print(f"FAISS index: {id_index.ntotal:,} vectors, dim={dim}")

# Save FAISS index
faiss_path = f"{OUT_DIR}/faiss/index.faiss"
faiss.write_index(id_index, faiss_path)

# Save node_id list (maps FAISS ordinal → graph node ID string)
node_id_map_path = f"{OUT_DIR}/faiss/node_id_map.json"
with open(node_id_map_path, "w") as f:
    json.dump(node_ids, f)

print(f"FAISS index  → {faiss_path}")
print(f"Node ID map  → {node_id_map_path}")

FAISS index: 10,713 vectors, dim=384
FAISS index  → /content/rag_artifacts/faiss/index.faiss
Node ID map  → /content/rag_artifacts/faiss/node_id_map.json


## Step 7 — Save Metadata Manifest
A small JSON file the Space app loads at startup to know what's available.

In [ ]:
manifest = {
    "versions": versions,
    "total_chunks": len(all_chunks),
    "total_nodes": G.number_of_nodes(),
    "total_edges": G.number_of_edges(),
    "embed_model": EMBED_MODEL,
    "embed_dim": dim,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "files": {
        "chunks": "data/chunks/chunks.jsonl",
        "graph":  "data/kg/graph.pkl",
        "entity_map": "data/kg/entity_to_chunks.json",
        "faiss_index": "data/faiss/index.faiss",
        "node_id_map": "data/faiss/node_id_map.json",
    }
}

manifest_path = f"{OUT_DIR}/manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest:")
print(json.dumps(manifest, indent=2))

Manifest:
{
  "versions": [
    "3.5",
    "3.4"
  ],
  "total_chunks": 10713,
  "total_nodes": 29165,
  "total_edges": 143063,
  "embed_model": "BAAI/bge-small-en-v1.5",
  "embed_dim": 384,
  "chunk_size": 400,
  "chunk_overlap": 60,
  "files": {
    "chunks": "data/chunks/chunks.jsonl",
    "graph": "data/kg/graph.pkl",
    "entity_map": "data/kg/entity_to_chunks.json",
    "faiss_index": "data/faiss/index.faiss",
    "node_id_map": "data/faiss/node_id_map.json"
  }
}


## Step 8 — Upload to HuggingFace Space Storage

Uses huggingface_hub to upload each artifact to the Space's persistent storage. Files land at data/ inside the Space's /data mount.

In [ ]:
api = HfApi(token=os.environ["HF_TOKEN"])

# Files to upload: (local_path, path_in_repo)
upload_manifest = [
    (manifest_path,      f"{HF_DATA_DIR}/manifest.json"),
    (chunks_path,        f"{HF_DATA_DIR}/chunks/chunks.jsonl"),
    (kg_path,            f"{HF_DATA_DIR}/kg/graph.pkl"),
    (e2c_path,           f"{HF_DATA_DIR}/kg/entity_to_chunks.json"),
    (faiss_path,         f"{HF_DATA_DIR}/faiss/index.faiss"),
    (node_id_map_path,   f"{HF_DATA_DIR}/faiss/node_id_map.json"),
]

print(f"Uploading {len(upload_manifest)} files to {HF_REPO_ID} ({HF_REPO_TYPE})...\n")

for local_path, repo_path in upload_manifest:
    size_mb = os.path.getsize(local_path) / 1e6
    print(f"  Uploading {repo_path} ({size_mb:.1f} MB)...", end=" ")
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=repo_path,
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        commit_message=f"[pipeline] Upload {os.path.basename(local_path)}",
    )
    print("done")

print("\nAll files uploaded successfully!")

Uploading 6 files to niclasfw/rubber-duck-games (space)...

  Uploading data/manifest.json (0.0 MB)... done
  Uploading data/chunks/chunks.jsonl (6.2 MB)... done
  Uploading data/kg/graph.pkl (12.7 MB)... 

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ag_artifacts/kg/graph.pkl:  10%|#         | 1.30MB / 12.7MB            

done
  Uploading data/kg/entity_to_chunks.json (0.4 MB)... done
  Uploading data/faiss/index.faiss (16.5 MB)... 

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tifacts/faiss/index.faiss:  35%|###4      | 5.78MB / 16.5MB            

done
  Uploading data/faiss/node_id_map.json (0.2 MB)... done

All files uploaded successfully!


## ✅ Done!

Your HuggingFace Space storage now contains:

```
data/
  manifest.json              ← metadata + file registry
  chunks/
    chunks.jsonl             ← all text chunks with metadata
  kg/
    graph.pkl                ← NetworkX DiGraph (nodes: doc/section/chunk/entity)
    entity_to_chunks.json    ← fast entity → chunk_id[] lookup
  faiss/
    index.faiss              ← cosine-similarity vector index (IndexIDMap)
    node_id_map.json         ← ordinal → graph node ID string
```

# Finetuning SmolLM3-3B for Coding

In [2]:
%%capture
!pip install modal trl

In [6]:
import os
from google.colab import userdata

In [ ]:
os.environ["MODAL_TOKEN_ID"] = "YOUR_TOKEN"
os.environ["MODAL_TOKEN_SECRET"] = "YOUR_SECRET"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [9]:
# @title
%%writefile finetune.py
import modal
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig


image = (
    modal.Image.debian_slim(python_version="3.12")
    .run_commands("apt-get update && apt-get install -y git")
    .pip_install(
        "torch==2.5.1",
        extra_index_url="https://download.pytorch.org/whl/cu121",
    )
    .pip_install(
        "trl==0.15.2",
        "peft==0.14.0",
        "bitsandbytes==0.46.1",
        "datasets==3.6.0",
        "accelerate==1.6.0",
        "huggingface_hub==0.32.0",
        "hf-transfer>=0.1.8",
        "hf_xet",
    )
    .pip_install(
        "git+https://github.com/huggingface/transformers.git",
    )
    .env({
        "HF_XET_HIGH_PERFORMANCE": "1",
        "TOKENIZERS_PARALLELISM": "false",
    })
)

app = modal.App("smollm3-codex-finetune")
volume = modal.Volume.from_name("smollm3-finetune-vol", create_if_missing=True)

VOLUME_PATH    = "/vol"
CHECKPOINT_DIR = f"{VOLUME_PATH}/checkpoints"
MODEL_ID       = "HuggingFaceTB/SmolLM3-3B"
DATASET_ID     = "Modotte/CodeX-7M-Non-Thinking"
HF_REPO_ID     = "niclasfw/smollm3-3b-codex"
MAX_STEPS      = 2000
BATCH_SIZE     = 4
GRAD_ACCUM     = 4
LR             = 2e-4
MAX_SEQ_LEN    = 2048
LORA_R         = 64
LORA_ALPHA     = 128
LORA_DROPOUT   = 0.05
NUM_SAMPLES    = 200_000


@app.function(
    image=image,
    gpu="A100-80GB",
    timeout=60 * 60 * 6,
    secrets=[modal.Secret.from_name("hf-secret")],
    volumes={VOLUME_PATH: volume},
)
def train():
    hf_token = os.environ["HF_TOKEN"]

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        token=hf_token,
    )
    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    dataset = load_dataset(DATASET_ID, split="train", token=hf_token)
    if NUM_SAMPLES:
        dataset = dataset.shuffle(seed=42).select(range(NUM_SAMPLES))

    def format_example(example):
        text = f"### Instruction:\n{example['input']}\n\n### Response:\n{example['output']}{tokenizer.eos_token}"
        return {"text": text}

    dataset = dataset.map(format_example, remove_columns=dataset.column_names)

    training_args = SFTConfig(
        output_dir=CHECKPOINT_DIR,
        max_steps=MAX_STEPS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        gradient_checkpointing=True,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        bf16=True,
        logging_steps=50,
        save_steps=500,
        save_total_limit=2,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=True,
        dataloader_num_workers=4,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        tokenizer=tokenizer,
    )
    trainer.train()

    # Save adapter only (skip merge to avoid bnb compatibility issue)
    trainer.model.save_pretrained(f"{VOLUME_PATH}/lora_adapter")
    tokenizer.save_pretrained(f"{VOLUME_PATH}/lora_adapter")
    print("✅ Adapter saved to volume")
    volume.commit()


@app.function(
    image=image,
    gpu="A100-80GB",
    timeout=60 * 60 * 2,
    secrets=[modal.Secret.from_name("hf-secret")],
    volumes={VOLUME_PATH: volume},
)
def merge_and_push():
    import os
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    hf_token = os.environ["HF_TOKEN"]

    print("Loading base model in bf16 for clean merge...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        token=hf_token,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        f"{VOLUME_PATH}/lora_adapter",
        token=hf_token,
    )

    print("Loading adapter...")
    model = PeftModel.from_pretrained(
        base_model,
        f"{VOLUME_PATH}/lora_adapter",
    )

    print("Merging...")
    merged_model = model.merge_and_unload()

    print("Pushing to HuggingFace...")
    merged_model.push_to_hub(HF_REPO_ID, token=hf_token, private=False)
    tokenizer.push_to_hub(HF_REPO_ID, token=hf_token, private=False)
    print(f"✅ Model pushed to https://huggingface.co/{HF_REPO_ID}")


@app.local_entrypoint()
def main():
    call = train.spawn()
    print(f"Training job spawned! Call ID: {call.object_id}")
    print(f"Monitor at: https://modal.com/apps")
    print("Once training completes, run: modal run finetune.py::merge_and_push")

Overwriting finetune.py


In [ ]:
!MODAL_TOKEN_ID="YOUR_TOKEN" MODAL_TOKEN_SECRET="YOUR_SECRET" modal run --detach finetune.py




Note that running a local entrypoint in detached mode only keeps the last triggered Modal function alive after the parent process has been killed or disconnected.
✓ Initialized. View run at 
https://modal.com/apps/niclas-wiegleb/main/ap-EAFtbPEkyLiW8W7m9s8MzD
⠋ Initializing...
⠸ Creating objects...
⠦ Creating objects...
├── 🔨 Created mount /content/finetune.py
├── ⠋ Creating function train...
⠧ Creating objects...
├── 🔨 Created mount /content/finetune.py
├── 🔨 Created function train.
└── 🔨 Created function merge_and_push.
✓ Created objects.
├── 🔨 Created mount /content/finetune.py
├── 🔨 Created function train.
└── 🔨 Created function merge_and_push.
Training job spawned! Call ID: fc-01KTMXM12GV3068ZM6ZSBHCG5Q
Monitor at: https://modal.com/apps
Once training completes, run: modal run finetune.py::merge_and_push
⠙ Running app...
[modal-client] 2026-06-09T00:48:28+0000 Timed out waiting for final app logs.
✓ App completed. View run at 
https://modal.com/apps/niclas-wiegleb/main/ap-EAFtb

In [ ]:
!MODAL_TOKEN_ID="YOUR_TOKEN" MODAL_TOKEN_SECRET="YOUR_SECRET" modal run finetune.py::merge_and_push

✓ Initialized. View run at 
https://modal.com/apps/niclas-wiegleb/main/ap-OLvUIqPRvFpEUrQtwEijCo
⠋ Initializing...
⠸ Creating objects...
⠦ Creating objects...
⠇ Creating objects...
├── 🔨 Created mount /content/finetune.py
├── 🔨 Created function merge_and_push.
└── 🔨 Created function train.
✓ Created objects.
├── 🔨 Created mount /content/finetune.py
├── 🔨 Created function merge_and_push.
└── 🔨 Created function train.
⠼ Worker assigned... View app at 
⠧ Worker assigned... View app at 
⠋ Worker assigned... View app at 
⠸ Running... View app at 
⠧ Running... View app at 
⠋ Running... View app at 
⠸ Loading images (1 containers initializing)... View app at 
⠦ Loading images (1 containers initializing)... View app at 
⠏ Loading images (1 containers initializing)... View app at 
⠹ Loading images (1 containers initializing)... View app at 
⠴ Loading images (1 containers initializing)... View app at 
⠇ Loading images (1 containers initializing)... View app at 
⠹ Loading images (1 containers ini